# 04 — Statistical Analysis
**NST DVA Capstone 2 · NASA Planetary Defense · Team SectionC_G-09**

## Objectives
- Compute correlation matrices for orbital parameters using descriptive column names
- Test statistical differences between PHAs and non-PHAs (t-tests, Mann-Whitney U)
- Analyse observation arc length and data quality by orbit class
- Identify KPIs to surface in the Tableau dashboard

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

PROCESSED_DIR = Path().resolve().parent / 'data' / 'processed'
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

nea   = pd.read_csv(PROCESSED_DIR / 'nea_catalogue_clean.csv', low_memory=False)
pha   = pd.read_csv(PROCESSED_DIR / 'nea_hazardous_clean.csv', low_memory=False)
close = pd.read_csv(PROCESSED_DIR / 'close_approaches_clean.csv')
print(f'nea:{nea.shape}  pha:{pha.shape}  close:{close.shape}')

## 4.1 — Orbital Parameter Correlation Matrix (NEA)

In [ ]:
orbital_cols = [
    'absolute_magnitude_h', 'orbital_eccentricity', 'semi_major_axis_au',
    'orbital_inclination_deg', 'perihelion_dist_au', 'aphelion_dist_au',
    'orbital_period_years', 'mean_motion_deg_per_day',
    'min_orbit_intersection_dist_au',
]
corr_cols = [c for c in orbital_cols if c in nea.columns]
corr = nea[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            xticklabels=[c.replace('_',' ') for c in corr.columns],
            yticklabels=[c.replace('_',' ') for c in corr.index], ax=ax)
ax.set_title('Orbital Parameter Correlation Matrix (NEA Catalogue)', fontsize=13, fontweight='bold')
plt.xticks(rotation=40, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

## 4.2 — Close Approach Variable Correlations

In [ ]:
close_cols = [
    'distance_au', 'distance_km', 'distance_lunar_distances',
    'velocity_km_s', 'velocity_relative_km_h', 'velocity_infinity_km_s',
    'absolute_magnitude',
]
corr_cols_c = [c for c in close_cols if c in close.columns]
corr_c = close[corr_cols_c].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask_c = np.triu(np.ones_like(corr_c, dtype=bool))
sns.heatmap(corr_c, mask=mask_c, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            xticklabels=[c.replace('_',' ') for c in corr_c.columns],
            yticklabels=[c.replace('_',' ') for c in corr_c.index], ax=ax)
ax.set_title('Close Approach Variable Correlations', fontsize=12, fontweight='bold')
plt.xticks(rotation=35, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

## 4.3 — PHA vs Non-PHA Statistical Comparison

In [ ]:
pha_mask = nea['is_potentially_hazardous'].astype(str).str.lower().isin(['true','1'])
pha_df   = nea[pha_mask]
non_pha  = nea[~pha_mask]

compare_cols = [
    ('absolute_magnitude_h',          'Absolute Magnitude H'),
    ('min_orbit_intersection_dist_au', 'MOID (AU)'),
    ('orbital_eccentricity',           'Orbital Eccentricity'),
    ('semi_major_axis_au',             'Semi-Major Axis (AU)'),
    ('orbital_inclination_deg',        'Inclination (deg)'),
    ('observation_span_years',         'Observation Span (years)'),
]

print(f"{'Metric':<35} {'PHA Mean':>10} {'Non-PHA Mean':>13} {'p-value':>10} {'Significant?':>13}")
print('─' * 83)
for col, label in compare_cols:
    if col not in nea.columns:
        continue
    g1 = pha_df[col].dropna()
    g2 = non_pha[col].dropna()
    _, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
    sig = '✓ Yes' if p < 0.05 else '  No'
    print(f'  {label:<33} {g1.mean():>10.3f} {g2.mean():>13.3f} {p:>10.2e} {sig:>13}')

## 4.4 — Boxplots: PHA vs Non-PHA Key Metrics

In [ ]:
plot_cols = [
    ('absolute_magnitude_h',          'Absolute Magnitude H'),
    ('min_orbit_intersection_dist_au', 'MOID (AU)'),
    ('orbital_eccentricity',           'Eccentricity'),
    ('orbital_inclination_deg',        'Inclination (deg)'),
]
plot_cols = [(c, l) for c, l in plot_cols if c in nea.columns]

fig, axes = plt.subplots(1, len(plot_cols), figsize=(14, 5))
for ax, (col, label) in zip(axes, plot_cols):
    nea['_pha_label'] = nea['is_potentially_hazardous'].astype(str).map({'True':'PHA','False':'Non-PHA'})
    sns.boxplot(data=nea, x='_pha_label', y=col, ax=ax,
                palette={'PHA':'crimson','Non-PHA':'steelblue'}, showfliers=False)
    ax.set(title=label, xlabel='', ylabel='')
plt.suptitle('PHA vs Non-PHA — Key Orbital Metrics (outliers hidden)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4.5 — Observation Data Quality by Orbit Class

In [ ]:
if 'orbit_class_label' in nea.columns and 'observation_span_years' in nea.columns:
    grp = (nea.groupby('orbit_class_label')['observation_span_years']
             .agg(['mean','median','count'])
             .sort_values('mean', ascending=False))
    print('Observation Span by Orbit Class:')
    print(grp.to_string(float_format='{:.1f}'.format))

    fig, ax = plt.subplots(figsize=(11, 5))
    grp['mean'].plot(kind='bar', ax=ax, color='teal', edgecolor='white')
    ax.set(title='Mean Observation Span (years) by Orbit Class',
           xlabel='Orbit Class', ylabel='Mean Observation Span (years)')
    plt.xticks(rotation=40, ha='right')
    plt.tight_layout()
    plt.show()

## 4.6 — Orbital Condition Code Distribution

In [ ]:
if 'orbital_condition_code' in nea.columns:
    code_counts = nea['orbital_condition_code'].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(code_counts.index.astype(str), code_counts.values, color='mediumpurple', edgecolor='white')
    ax.set(title='Orbital Condition Code Distribution\n(0 = best orbit quality, 9 = poorest)',
           xlabel='Orbital Condition Code', ylabel='Count')
    plt.tight_layout()
    plt.show()
    print(f'Median condition code: {nea["orbital_condition_code"].median()}')

## 4.7 — KPI Summary for Tableau

| KPI | Value | Source Column |
|---|---|---|
| Total NEAs | computed | `nea_catalogue_clean` |
| Total PHAs | computed | `is_potentially_hazardous` |
| Critical-tier PHAs | computed | `risk_tier == 'Critical'` |
| Median MOID (all) | computed | `min_orbit_intersection_dist_au` |
| Median MOID (PHAs) | computed | `min_orbit_intersection_dist_au` (PHA subset) |
| Total future approaches | computed | `close_approaches_future_clean` |
| Median approach velocity | computed | `velocity_km_s` |
| Very close approaches (<10 LD) | computed | `is_very_close_approach` |

**Next → Notebook 05: Final Load Prep & Validation**